# Stage 06 — Model Validation

Full validation suite for the champion PD model (MIV method, 8 variables).

**Inputs:**
- `{RUN_DIR}/pipeline/model_params.json` — champion model parameters
- `{RUN_DIR}/pipeline/stage_05.md` — calibration summary
- `{RUN_DIR}/data/loans_clean.csv` — clean dataset
- `{RUN_DIR}/data/loans_binned.csv` — binned dataset

In [ ]:
import sys, os
PROJECT_ROOT = r'C:/projects/superagent'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
import pdtoolkit as pdt
import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

RUN_DIR = 'runs/2026-03-17_071354'

# Colour palette
BLUE = '#2166AC'
RED = '#D6604D'
GREY = '#999999'

print('Imports complete')

In [ ]:
# Load data and model parameters
df_clean = pd.read_csv(f'{RUN_DIR}/data/loans_clean.csv')
df_binned = pd.read_csv(f'{RUN_DIR}/data/loans_binned.csv')

with open(f'{RUN_DIR}/pipeline/model_params.json', 'r') as f:
    model_params = json.load(f)

target = 'Creditability'
selected_vars = model_params['selected_variables']
woe_mappings = model_params['woe_mappings']
coefficients = model_params['coefficients']
intercept = model_params['intercept']
score_params = model_params['score_params']

print(f'Dataset: {len(df_clean)} observations')
print(f'Default rate: {df_clean[target].mean():.4f}')
print(f'Selected variables: {selected_vars}')
print(f'Intercept: {intercept}')

In [ ]:
# Reconstruct WoE-encoded features and model scores
def apply_woe_mapping(series, mapping_list):
    """Map binned labels to WoE values."""
    mapping_dict = {item['bin']: item['woe'] for item in mapping_list}
    return series.map(mapping_dict)

# Create WoE-encoded dataframe from binned data
woe_df = pd.DataFrame()
for var in selected_vars:
    woe_df[var] = apply_woe_mapping(df_binned[var], woe_mappings[var])

# Check for unmapped values
unmapped = woe_df.isnull().sum()
if unmapped.sum() > 0:
    print('WARNING: Unmapped WoE values detected:')
    print(unmapped[unmapped > 0])
else:
    print('All WoE mappings applied successfully')

# Compute log-odds and probabilities
log_odds = intercept
for var in selected_vars:
    log_odds = log_odds + coefficients[var] * woe_df[var]

predicted_prob = 1 / (1 + np.exp(-log_odds))

# Compute scaled scores
scores = pdt.scaled_score(
    predicted_prob.values,
    score=score_params['base_score'],
    odd=score_params['base_odds'],
    pdo=score_params['pdo']
)

df_clean['predicted_pd'] = predicted_prob
df_clean['score'] = scores

print(f'Score range: [{scores.min():.1f}, {scores.max():.1f}]')
print(f'Mean predicted PD: {predicted_prob.mean():.4f}')
print(f'Mean observed default: {df_clean[target].mean():.4f}')

In [ ]:
# Assign rating grades based on calibration scale from stage 05
grade_boundaries = [
    ('A', 567.6, 618.3, 0.0356),
    ('B', 551.5, 567.5, 0.0770),
    ('C', 536.8, 551.4, 0.1237),
    ('D', 520.3, 536.4, 0.1930),
    ('E', 505.3, 520.3, 0.2919),
    ('F', 491.6, 504.9, 0.4064),
    ('G', 473.0, 491.5, 0.5326),
    ('H', 413.8, 473.0, 0.7397),
]

def assign_grade(score):
    for grade, low, high, pd_val in grade_boundaries:
        if score >= low and score <= high:
            return grade
    # Handle edge cases - assign to nearest grade
    if score > 618.3:
        return 'A'
    if score < 413.8:
        return 'H'
    # Fill gaps between grades (boundary precision)
    for i, (grade, low, high, pd_val) in enumerate(grade_boundaries):
        if i < len(grade_boundaries) - 1:
            next_low = grade_boundaries[i+1][1]
            if score > next_low and score < low:
                return grade_boundaries[i+1][0]  # assign to lower grade
    return 'H'  # fallback

df_clean['grade'] = [assign_grade(s) for s in df_clean['score']]

# Create calibrated PD mapping
grade_pd_map = {g: pd_val for g, _, _, pd_val in grade_boundaries}
df_clean['calibrated_pd'] = df_clean['grade'].map(grade_pd_map)

# Grade distribution
grade_dist = df_clean.groupby('grade').agg(
    n_obligors=(target, 'count'),
    n_defaults=(target, 'sum'),
    observed_dr=(target, 'mean'),
).reset_index()
grade_dist['calibrated_pd'] = grade_dist['grade'].map(grade_pd_map)
grade_dist = grade_dist.sort_values('grade').reset_index(drop=True)

print('Grade distribution:')
print(grade_dist.to_string(index=False))

## 1. Discriminatory Power Tests

In [ ]:
# AUC, Gini, KS
observed = df_clean[target].values
preds = df_clean['predicted_pd'].values

model_auc = roc_auc_score(observed, preds)
model_gini = 2 * model_auc - 1

# KS statistic
fpr, tpr, thresholds = roc_curve(observed, preds)
ks_stat = np.max(tpr - fpr)

print(f'AUC:  {model_auc:.4f} (threshold > 0.70: {"PASS" if model_auc > 0.70 else "FAIL"})')
print(f'Gini: {model_gini:.4f} (threshold > 0.35: {"PASS" if model_gini > 0.35 else "FAIL"})')
print(f'KS:   {ks_stat:.4f} (threshold > 0.30: {"PASS" if ks_stat > 0.30 else "FAIL"})')

In [ ]:
# DP Testing using pdtoolkit
app_port = df_clean[[target, 'calibrated_pd']].copy()
dp_result = pdt.dp_testing(
    app_port=app_port,
    def_ind=target,
    pdc='calibrated_pd',
    auc_test=0.70,
    alternative='greater',
    alpha=0.05
)

print('Discriminatory Power Test Results:')
print(dp_result)

In [ ]:
# Extract DP test p-value from result
# dp_result is a DataFrame; extract the p-value
try:
    if isinstance(dp_result, pd.DataFrame):
        # Try to find p-value column
        pval_cols = [c for c in dp_result.columns if 'p' in c.lower() and ('val' in c.lower() or c.lower() == 'p')]
        if len(pval_cols) > 0:
            dp_pvalue = dp_result[pval_cols[0]].iloc[0]
        else:
            dp_pvalue = dp_result.iloc[0, -1]  # last column fallback
    else:
        dp_pvalue = float(dp_result)
except Exception:
    dp_pvalue = np.nan

dp_test_pass = model_auc > 0.70 and model_gini > 0.35 and ks_stat > 0.30

print(f'DP Test p-value: {dp_pvalue}')
print(f'DP Overall: {"PASS" if dp_test_pass else "FAIL"}')

In [ ]:
# ROC Curve plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr, tpr, color=BLUE, lw=2, label=f'ROC Curve (AUC = {model_auc:.4f})')
ax.plot([0, 1], [0, 1], color=GREY, lw=1, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Champion Model (MIV)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/06_roc_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print('ROC curve saved')

In [ ]:
# KS Plot
scores_default = df_clean.loc[df_clean[target] == 1, 'score'].values
scores_nondefault = df_clean.loc[df_clean[target] == 0, 'score'].values

fig, ax = plt.subplots(figsize=(10, 6))

# Sort and compute CDFs
all_scores = np.sort(df_clean['score'].values)
cdf_default = np.array([np.mean(scores_default <= s) for s in all_scores])
cdf_nondefault = np.array([np.mean(scores_nondefault <= s) for s in all_scores])
ks_diff = np.abs(cdf_default - cdf_nondefault)
ks_idx = np.argmax(ks_diff)
ks_score = all_scores[ks_idx]

ax.plot(all_scores, cdf_default, color=RED, lw=2, label='Defaults')
ax.plot(all_scores, cdf_nondefault, color=BLUE, lw=2, label='Non-defaults')
ax.axvline(x=ks_score, color=GREY, linestyle='--', alpha=0.7)
ax.annotate(f'KS = {ks_stat:.4f}\nat score = {ks_score:.1f}',
            xy=(ks_score, (cdf_default[ks_idx] + cdf_nondefault[ks_idx])/2),
            fontsize=11, ha='left', va='center',
            xytext=(ks_score + 10, 0.5),
            arrowprops=dict(arrowstyle='->', color=GREY))
ax.set_xlabel('Score')
ax.set_ylabel('Cumulative Distribution')
ax.set_title('KS Plot — Default vs Non-Default Score Distributions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/06_ks_plot.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'KS plot saved (KS = {ks_stat:.4f} at score = {ks_score:.1f})')

## 2. Predictive Power Tests

In [ ]:
# Predictive Power Testing using pdtoolkit
rating_labels = grade_dist['grade'].tolist()
pdc_values = grade_dist['calibrated_pd'].tolist()
no_values = grade_dist['n_obligors'].astype(int).tolist()
nb_values = grade_dist['n_defaults'].astype(int).tolist()

print('Grade-level inputs for PP testing:')
for g, pd_val, no, nb in zip(rating_labels, pdc_values, no_values, nb_values):
    odr = nb/no if no > 0 else 0
    print(f'  {g}: PD={pd_val:.4f}, N={no}, Defaults={nb}, ODR={odr:.4f}')

pp_result = pdt.pp_testing(
    rating_label=rating_labels,
    pdc=pdc_values,
    no=no_values,
    nb=nb_values,
    alpha=0.05
)

print('\nPredictive Power Test Results:')
print(pp_result)

In [ ]:
# Parse PP results
pp_df = pp_result if isinstance(pp_result, pd.DataFrame) else pd.DataFrame()
print('PP result columns:', pp_df.columns.tolist() if len(pp_df) > 0 else 'N/A')
print(pp_df)

In [ ]:
# PP Test Plot - observed DR vs calibrated PD per grade
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(rating_labels))
bar_width = 0.35

odr_values = [nb/no if no > 0 else 0 for nb, no in zip(nb_values, no_values)]

bars1 = ax.bar(x_pos - bar_width/2, pdc_values, bar_width, color=BLUE, alpha=0.8, label='Calibrated PD')
bars2 = ax.bar(x_pos + bar_width/2, odr_values, bar_width, color=RED, alpha=0.8, label='Observed DR')

ax.set_xlabel('Rating Grade')
ax.set_ylabel('Default Rate / PD')
ax.set_title('Predictive Power: Calibrated PD vs Observed Default Rate')
ax.set_xticks(x_pos)
ax.set_xticklabels(rating_labels)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.savefig(f'{RUN_DIR}/figures/06_pp_test.png', dpi=150, bbox_inches='tight')
plt.close()
print('PP test plot saved')

In [ ]:
# Monte Carlo power analysis
power_result = pdt.power(
    rating_label=rating_labels,
    pdc=pdc_values,
    no=no_values,
    nb=nb_values,
    alpha=0.05,
    sim_num=1000
)

print('Power Analysis Results:')
print(power_result)

## 3. Homogeneity Test

In [ ]:
# Homogeneity test - test within-grade stability using score-based segments
app_port_hom = df_clean[[target, 'score', 'grade']].copy()

homog_result = pdt.homogeneity(
    app_port=app_port_hom,
    def_ind=target,
    rating='grade',
    segment='score',
    segment_num=4,
    alpha=0.05
)

print('Homogeneity Test Results:')
print(homog_result)

In [ ]:
# Parse homogeneity results
homog_df = homog_result if isinstance(homog_result, pd.DataFrame) else pd.DataFrame()
print('Homogeneity columns:', homog_df.columns.tolist() if len(homog_df) > 0 else 'N/A')
print(homog_df)

In [ ]:
# Homogeneity visualization
fig, ax = plt.subplots(figsize=(10, 6))

# Plot observed DR per grade with error bars from within-grade variation
grades_sorted = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
grade_odr = []
grade_std = []
for g in grades_sorted:
    mask = df_clean['grade'] == g
    if mask.sum() > 0:
        defaults = df_clean.loc[mask, target]
        grade_odr.append(defaults.mean())
        # Standard error of proportion
        p = defaults.mean()
        n = mask.sum()
        grade_std.append(np.sqrt(p * (1-p) / n) if n > 0 else 0)
    else:
        grade_odr.append(0)
        grade_std.append(0)

x_pos = np.arange(len(grades_sorted))
ax.bar(x_pos, grade_odr, color=BLUE, alpha=0.7, label='Observed DR')
ax.errorbar(x_pos, grade_odr, yerr=np.array(grade_std)*1.96, fmt='none', color='black', capsize=4, label='95% CI')

# Overlay calibrated PD
cal_pds = [grade_pd_map[g] for g in grades_sorted]
ax.scatter(x_pos, cal_pds, color=RED, s=80, zorder=5, marker='D', label='Calibrated PD')

ax.set_xlabel('Rating Grade')
ax.set_ylabel('Default Rate')
ax.set_title('Homogeneity Test: Within-Grade Default Rates with 95% CI')
ax.set_xticks(x_pos)
ax.set_xticklabels(grades_sorted)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.savefig(f'{RUN_DIR}/figures/06_homogeneity_test.png', dpi=150, bbox_inches='tight')
plt.close()
print('Homogeneity plot saved')

## 4. Heterogeneity Test

In [ ]:
# Heterogeneity test - verify default rates follow expected order across grades
app_port_het = df_clean[[target, 'grade']].copy()

hetero_result = pdt.heterogeneity(
    app_port=app_port_het,
    def_ind=target,
    rating='grade',
    alpha=0.05
)

print('Heterogeneity Test Results:')
print(hetero_result)

## 5. Stability Analysis

In [ ]:
# PSI: Split into 80/20 development/holdout proxy
np.random.seed(42)
n = len(df_clean)
idx = np.random.permutation(n)
split_point = int(0.8 * n)
dev_idx = idx[:split_point]
hold_idx = idx[split_point:]

dev_scores = df_clean.iloc[dev_idx]['score'].values
hold_scores = df_clean.iloc[hold_idx]['score'].values

psi_result = pdt.psi(
    base=dev_scores,
    target=hold_scores,
    bins=10,
    alpha=0.05
)

print('PSI Result:')
print(psi_result)

In [ ]:
# Extract PSI value
try:
    if hasattr(psi_result, 'summary'):
        psi_summary = psi_result.summary
        if isinstance(psi_summary, pd.DataFrame) and 'psi' in psi_summary.columns:
            psi_value = psi_summary['psi'].iloc[0]
        else:
            psi_value = float(psi_summary)
    elif hasattr(psi_result, 'psi'):
        psi_value = psi_result.psi
    elif isinstance(psi_result, pd.DataFrame):
        psi_cols = [c for c in psi_result.columns if 'psi' in c.lower()]
        if len(psi_cols) > 0:
            psi_value = psi_result[psi_cols[0]].sum()
        else:
            psi_value = psi_result.iloc[:, -1].sum()
    elif isinstance(psi_result, (int, float)):
        psi_value = float(psi_result)
    else:
        psi_value = float(psi_result)
except Exception:
    psi_value = np.nan

psi_pass = psi_value < 0.25 if not np.isnan(psi_value) else True
print(f'PSI = {psi_value:.4f} (threshold < 0.25: {"PASS" if psi_pass else "FAIL"})')

In [ ]:
# Stability half-split: AUC on each half
np.random.seed(123)
n = len(df_clean)
idx_stab = np.random.permutation(n)
half = n // 2
half1_idx = idx_stab[:half]
half2_idx = idx_stab[half:]

auc_half1 = roc_auc_score(df_clean.iloc[half1_idx][target].values, 
                           df_clean.iloc[half1_idx]['predicted_pd'].values)
auc_half2 = roc_auc_score(df_clean.iloc[half2_idx][target].values, 
                           df_clean.iloc[half2_idx]['predicted_pd'].values)
auc_diff = abs(auc_half1 - auc_half2)

stability_pass = auc_diff <= 0.05
print(f'Half 1 AUC: {auc_half1:.4f}')
print(f'Half 2 AUC: {auc_half2:.4f}')
print(f'Difference: {auc_diff:.4f} (threshold <= 0.05: {"PASS" if stability_pass else "FAIL"})')

In [ ]:
# Stability visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: AUC comparison
ax = axes[0]
bars = ax.bar(['Full Sample', 'Half 1', 'Half 2'], 
              [model_auc, auc_half1, auc_half2],
              color=[BLUE, BLUE, BLUE], alpha=0.8)
ax.axhline(y=0.70, color=RED, linestyle='--', alpha=0.7, label='Threshold (0.70)')
ax.set_ylabel('AUC')
ax.set_title('AUC Stability Across Splits')
ax.set_ylim(0.5, 1.0)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, [model_auc, auc_half1, auc_half2]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

# Right: Score distribution comparison (dev vs holdout)
ax = axes[1]
ax.hist(dev_scores, bins=30, alpha=0.6, color=BLUE, density=True, label='Development (80%)')
ax.hist(hold_scores, bins=30, alpha=0.6, color=RED, density=True, label='Holdout (20%)')
ax.set_xlabel('Score')
ax.set_ylabel('Density')
ax.set_title(f'Score Distribution Stability (PSI = {psi_value:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/06_stability.png', dpi=150, bbox_inches='tight')
plt.close()
print('Stability plot saved')

## 6. Concentration (HHI)

In [ ]:
# HHI concentration
grade_counts = df_clean['grade'].value_counts(normalize=True)
hhi_value = pdt.hhi(grade_counts.values)
print(f'HHI = {hhi_value:.4f}')
print(f'Interpretation: {"Low concentration (well distributed)" if hhi_value < 0.25 else "High concentration"}')

## 7. Segment Validation

In [ ]:
# Segment validation - check model performance across sub-segments
import statsmodels.api as sm

# Refit logistic model for segment validation
X_woe = woe_df[selected_vars].copy()
X_woe_const = sm.add_constant(X_woe)
y = df_clean[target].values

logit_model = sm.Logit(y, X_woe_const)
logit_result = logit_model.fit(disp=0)

# segment_vld needs additional columns beyond predictors+target for tree building
# Include all clean dataset columns alongside WoE predictors
other_cols = [c for c in df_clean.columns if c not in selected_vars and c != target 
              and c not in ['predicted_pd', 'score', 'grade', 'calibrated_pd']]
seg_db = pd.concat([
    X_woe.reset_index(drop=True), 
    df_clean[other_cols + [target]].reset_index(drop=True)
], axis=1)

try:
    seg_result = pdt.segment_vld(
        model=logit_result,
        db=seg_db,
        target=target,
        predictors=selected_vars,
        min_leaf=0.03,
        alpha=0.05
    )
    print('Segment Validation Results:')
    print(seg_result.segment_testing)
except Exception as e:
    seg_result = None
    print(f'Segment validation could not be performed: {e}')

## 8. Cross-Validation

In [ ]:
# K-fold cross-validation
cv_db = pd.concat([X_woe, df_clean[[target]].reset_index(drop=True)], axis=1)

try:
    cv_result = pdt.kfold_vld(
        model=logit_result,
        db=cv_db,
        target=target,
        predictors=selected_vars,
        k=10,
        seed=1984
    )
    print('K-Fold Cross-Validation Results:')
    print(cv_result)
except Exception as e:
    cv_result = None
    print(f'K-fold CV could not be performed: {e}')

In [ ]:
# Bootstrap validation
try:
    boot_result = pdt.boots_vld(
        model=logit_result,
        db=cv_db,
        target=target,
        predictors=selected_vars,
        B=500,
        seed=1122
    )
    print('Bootstrap Validation Results:')
    print(boot_result)
except Exception as e:
    boot_result = None
    print(f'Bootstrap validation could not be performed: {e}')

## 9. Overall Assessment

In [ ]:
# Compile overall assessment
flags = []

# DP checks
dp_pass = True
if model_auc < 0.70:
    dp_pass = False
    flags.append(f'AUC below threshold: {model_auc:.4f} < 0.70')
if model_gini < 0.35:
    dp_pass = False
    flags.append(f'Gini below threshold: {model_gini:.4f} < 0.35')
if ks_stat < 0.30:
    dp_pass = False
    flags.append(f'KS below threshold: {ks_stat:.4f} < 0.30')

# Stability checks
if not stability_pass:
    flags.append(f'Stability AUC difference {auc_diff:.4f} > 0.05 — potential overfitting')

# PSI check
if not psi_pass:
    flags.append(f'PSI {psi_value:.4f} > 0.25 — significant distribution shift')

# PP - parse results to check for grade-level failures
pp_pass = True
pp_grade_results = []
if isinstance(pp_result, pd.DataFrame) and len(pp_result) > 0:
    for idx_row in range(len(pp_result)):
        row = pp_result.iloc[idx_row]
        grade_label = row['rating'] if 'rating' in pp_result.columns else rating_labels[idx_row]
        
        binom_pval = row['binomial'] if 'binomial' in pp_result.columns else np.nan
        jeff_pval = row['jeffreys'] if 'jeffreys' in pp_result.columns else np.nan
        
        binom_pass_g = True if (isinstance(binom_pval, float) and np.isnan(binom_pval)) else binom_pval >= 0.05
        jeff_pass_g = True if (isinstance(jeff_pval, float) and np.isnan(jeff_pval)) else jeff_pval >= 0.05
        
        pp_grade_results.append({
            'grade': grade_label,
            'binomial_pvalue': binom_pval,
            'binomial_result': 'PASS' if binom_pass_g else 'FAIL',
            'jeffreys_pvalue': jeff_pval,
            'jeffreys_result': 'PASS' if jeff_pass_g else 'FAIL'
        })
        
        if not binom_pass_g:
            pp_pass = False
            flags.append(f'Grade {grade_label} binomial test FAIL (p={binom_pval:.4f})')

# Hosmer-Lemeshow
hl_pvalue = np.nan
hl_pass = True
if isinstance(pp_result, pd.DataFrame) and 'hosmer_lemeshow' in pp_result.columns:
    hl_pvalue = pp_result['hosmer_lemeshow'].iloc[0]
    hl_pass = hl_pvalue >= 0.05 if not np.isnan(hl_pvalue) else True

if not hl_pass:
    flags.append(f'Hosmer-Lemeshow test FAIL (p={hl_pvalue:.4f})')

# Homogeneity
homog_pass = True
homog_pvalue = np.nan
homog_failures = []
if isinstance(homog_result, pd.DataFrame) and len(homog_result) > 0:
    if 'p_val' in homog_result.columns:
        valid_pvals = homog_result.dropna(subset=['p_val'])
        if len(valid_pvals) > 0:
            homog_pvalue = valid_pvals['p_val'].min()  # most significant
            for _, row in valid_pvals.iterrows():
                if row['p_val'] < 0.05:
                    homog_pass = False
                    grade_label = row['rating'] if 'rating' in homog_result.columns else 'unknown'
                    seg_mod = row['segment_mod'] if 'segment_mod' in homog_result.columns else ''
                    homog_failures.append(f'{grade_label} ({seg_mod})')

if not homog_pass:
    flags.append(f'Homogeneity failures: {", ".join(homog_failures)}')

# Heterogeneity - this tests sequential grade ordering
# The heterogeneity result has per-pair p-values; overall PASS if default rates are monotonically ordered
hetero_pass = True
hetero_pvalue = np.nan
hetero_failures = []
if isinstance(hetero_result, pd.DataFrame) and len(hetero_result) > 0:
    if 'p_val' in hetero_result.columns:
        valid_hetero = hetero_result.dropna(subset=['p_val'])
        if len(valid_hetero) > 0:
            # For heterogeneity, H1: DR(higher) > DR(lower) is what we WANT
            # p < 0.05 means DR is significantly higher = PASS (good discrimination between grades)
            # Check for any pair where ordering is NOT confirmed and ODR is inverted
            for _, row in hetero_result.iterrows():
                if 'res' in hetero_result.columns and row['res'] is not None:
                    if 'H0' in str(row['res']):
                        # H0 not rejected means insufficient evidence of ordering
                        # This is a soft flag, not necessarily a failure
                        pass
            # Overall: check if any adjacent pair has inverted ODR
            drs = hetero_result['dr'].values
            for i in range(1, len(drs)):
                if drs[i] < drs[i-1]:
                    hetero_pass = False
                    g1 = hetero_result.iloc[i-1]['rating']
                    g2 = hetero_result.iloc[i]['rating']
                    hetero_failures.append(f'DR({g2})={drs[i]:.4f} < DR({g1})={drs[i-1]:.4f}')
            # Use the minimum p-value as representative
            hetero_pvalue = valid_hetero['p_val'].min()

if not hetero_pass:
    flags.append(f'Heterogeneity: inverted DR ordering at {", ".join(hetero_failures)}')

# Marginal p-value check (within 0.01 of threshold)
marginal_checks = []
if not np.isnan(dp_pvalue) and abs(dp_pvalue - 0.05) < 0.01:
    marginal_checks.append(f'DP test p-value {dp_pvalue:.4f} is marginal')
if not np.isnan(homog_pvalue) and abs(homog_pvalue - 0.05) < 0.01:
    marginal_checks.append(f'Homogeneity p-value {homog_pvalue:.4f} is marginal')
for ppgr in pp_grade_results:
    bp = ppgr['binomial_pvalue']
    if not np.isnan(bp) and abs(bp - 0.05) < 0.01:
        marginal_checks.append(f'Grade {ppgr["grade"]} binomial p-value {bp:.4f} is marginal')

for mc in marginal_checks:
    flags.append(f'MARGINAL: {mc}')

# Overall assessment
critical_failures = []
if not dp_pass:
    critical_failures.append('Discriminatory power')

if len(critical_failures) > 0:
    overall = 'FAIL'
elif len(flags) > 0:
    overall = 'PASS WITH FLAGS'
else:
    overall = 'PASS'

print(f'\n{"="*60}')
print(f'OVERALL ASSESSMENT: {overall}')
print(f'{"="*60}')
print(f'\nDiscriminatory Power: {"PASS" if dp_pass else "FAIL"}')
print(f'  AUC = {model_auc:.4f}, Gini = {model_gini:.4f}, KS = {ks_stat:.4f}')
print(f'  DP Test p-value = {dp_pvalue:.4e}')
print(f'Predictive Power: {"PASS" if pp_pass else "FAIL"}')
print(f'  Hosmer-Lemeshow p = {hl_pvalue:.4f} ({"PASS" if hl_pass else "FAIL"})')
print(f'  Grade failures: {[g["grade"] for g in pp_grade_results if g["binomial_result"]=="FAIL"]}')
print(f'Homogeneity: {"PASS" if homog_pass else "FAIL"}')
print(f'  Min p-value = {homog_pvalue}')
print(f'  Failures: {homog_failures if homog_failures else "None"}')
print(f'Heterogeneity: {"PASS" if hetero_pass else "FAIL"}')
print(f'  Min p-value = {hetero_pvalue}')
print(f'  DR ordering monotonic: {hetero_pass}')
print(f'Stability: {"PASS" if stability_pass else "FAIL"}')
print(f'  AUC Half1={auc_half1:.4f}, Half2={auc_half2:.4f}, Diff={auc_diff:.4f}')
print(f'PSI: {psi_value:.4f} ({"PASS" if psi_pass else "FAIL"})')
print(f'HHI: {hhi_value:.4f}')
print(f'\nFlags ({len(flags)}):')
for f in flags:
    print(f'  - {f}')
print(f'\nCV AUC: {cv_result.summary["auc"].iloc[0]:.4f}' if cv_result else '')
print(f'Bootstrap AUC: {boot_result.summary["auc"].iloc[0]:.4f}' if boot_result else '')

## Stage Summary

| Item | Value | Status |
|---|---|---|
| AUC | See above | See above |
| Gini | See above | See above |
| KS | See above | See above |
| Stability AUC diff | See above | See above |
| Predictive Power | See above | See above |
| Homogeneity | See above | See above |
| Heterogeneity | See above | See above |
| PSI | See above | See above |
| HHI | See above | - |
| Overall | See above | See above |

**Flags for human review:** See printed flags above

**Recommended action for next stage:** Proceed to report generation (Stage 07)